# SIGMOD Exp 4: Snapshot Concurrency

This experiment has two parts. First, `snapshot_concurrency_bench` measures old-snapshot and fresh-read blocking directly and reports those values as a compact numeric table. Second, `htap_trace_bench` runs a 100-transaction mixed HTAP trace over join transactions, full derived-state scans, delta transactions over readable epochs, and interleaved `0.01%` update waves on a shared worker pool. Historical reads target only readable epoch timestamps, and EPOCH creates new partitions only at those readable boundaries.

1. Blocking summary table
2. Fixed mixed-trace completion time
3. Historical-percentage sweep of mixed-trace completion time (with delta fixed at 0 to isolate snapshot-cache effects)

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'pandas', 'matplotlib', 'numpy'])
print('done')

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('../../').resolve()
sys.path.append(str(ROOT / 'benches'))
import importlib
import sigmod_exp_common as _sigmod_exp_common
importlib.reload(_sigmod_exp_common)

from sigmod_exp_common import (
    TOL,
    SIGMOD_BUCKET_NUM,
    SIGMOD_FIXED_UPDATE_PCT,
    SIGMOD_REPEAT,
    SIGMOD_TPCH_SF,
    SIGMOD_TRIM,
    SIGMOD_UPDATE_SWEEP_PCTS,
    SIGMOD_WARMUP,
    apply_paper_style,
    ensure_dirs,
    resolve_tpch_file,
    resolve_update_file,
    run_checked,
)

apply_paper_style(ROOT)

EXP_DIR = (ROOT / 'benches' / 'sigmod_exp4_concurrency').resolve()
DATA_DIR = EXP_DIR / 'data'
FIGS_DIR = EXP_DIR / 'figs'
ensure_dirs(DATA_DIR, FIGS_DIR)

TPCH_DIR = (ROOT / 'benches' / 'sigmod' / 'tpch_data').resolve()
GEN_UPDATES = (ROOT / 'benches' / 'sigmod' / 'generate_updates.py').resolve()
SNAPSHOT_BIN = ROOT / 'target' / 'release' / 'snapshot_concurrency_bench'
TRACE_BIN = ROOT / 'target' / 'release' / 'htap_trace_bench'

SF = SIGMOD_TPCH_SF
BUCKET_NUM = SIGMOD_BUCKET_NUM
WARMUP = SIGMOD_WARMUP
REPEAT = SIGMOD_REPEAT
TRIM = SIGMOD_TRIM
READ_TX_SIZE = 2048
FIXED_UPDATE_PCT = SIGMOD_FIXED_UPDATE_PCT
FIXED_READER_THREADS = 4
TRACE_WORKER_THREADS = 5
TRACE_TOTAL_TXS = 100
TRACE_UPDATE_WAVES = 30
TRACE_READ_BUDGET = TRACE_TOTAL_TXS - TRACE_UPDATE_WAVES
TRACE_FIXED_DELTA_PCT = 10
TRACE_FIXED_HIST_PCT = 40
TRACE_READABLE_EVERY = 2
HIST_SWEEP_DELTA_PCT = 0
HIST_PCT_SWEEP = [0, 20, 40, 60, 80, 100]
SERIES = [
    ('snap', 'nr'),
    ('ivmh', 'nr'),
    ('heap', 'wr'),
    ('chain', 'wr'),
    ('par', 'wr'),
]
STYLE = {
    ('snap', 'nr'): ('SNAP', TOL['red'], ':', 'x'),
    ('ivmh', 'nr'): ('IVMH', TOL['yellow'], '--', 'P'),
    ('heap', 'wr'): ('MONO-WR', TOL['blue'], '-', 'o'),
    ('chain', 'wr'): ('DUAL-WR', TOL['cyan'], '-', 's'),
    ('par', 'wr'): ('EPOCH-WR', TOL['green'], '-', 'D'),
}


def normalize_result_df(df):
    df = df.copy()
    for col in ('table_type', 'repair_mode'):
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.lower()
    if 'historical_ratio' in df.columns:
        df['historical_pct'] = df['historical_ratio'].astype(float) * 100.0
    if 'delta_txs' in df.columns:
        df['delta_pct'] = df['delta_txs'].astype(float)
    return df


def trace_shape_from_delta_pct(delta_pct):
    delta_txs = int(round(delta_pct))
    if delta_txs < 0 or delta_txs > TRACE_READ_BUDGET:
        raise ValueError(f'invalid delta percentage {delta_pct}; expected 0..{TRACE_READ_BUDGET}')
    non_delta_reads = TRACE_READ_BUDGET - delta_txs
    join_txs = int(round(non_delta_reads * (2.0 / 3.0)))
    scan_txs = non_delta_reads - join_txs
    return join_txs, scan_txs, delta_txs


PART_FILE = resolve_tpch_file(TPCH_DIR, 'part', SF)
LINEITEM_FILE = resolve_tpch_file(TPCH_DIR, 'lineitem_probe', SF, '_1995-09-01_1995-10-01.tbl')

print('PART        :', PART_FILE)
print('LINEITEM    :', LINEITEM_FILE)
print('SNAPSHOT BIN:', SNAPSHOT_BIN)
print('TRACE BIN   :', TRACE_BIN)


In [ ]:
print('Building snapshot_concurrency_bench...')
run_checked(['cargo', 'build', '--release', '--bin', 'snapshot_concurrency_bench'], ROOT)
print('Building htap_trace_bench...')
run_checked(['cargo', 'build', '--release', '--bin', 'htap_trace_bench'], ROOT)
print('Build OK')

In [ ]:
def ensure_update_file(pct):
    try:
        return resolve_update_file(TPCH_DIR, SF, pct, 'uniform')
    except FileNotFoundError:
        print(f'Generating update file for {pct}%...')
        run_checked([sys.executable, str(GEN_UPDATES), str(PART_FILE), str(pct), str(SF), '--output-dir', str(TPCH_DIR)], ROOT)
        return resolve_update_file(TPCH_DIR, SF, pct, 'uniform')


def run_blocking_point(table_type, repair_mode, reader_threads, update_pct, output_csv):
    updates_file = ensure_update_file(update_pct)
    result = run_checked([
        str(SNAPSHOT_BIN),
        '--part-file', str(PART_FILE),
        '--lineitem-file', str(LINEITEM_FILE),
        '--updates-file', str(updates_file),
        '--table-type', table_type,
        '--repair-mode', repair_mode,
        '--bucket-num', str(BUCKET_NUM),
        '--reader-threads', str(reader_threads),
        '--read-tx-size', str(READ_TX_SIZE),
        '--warmup', str(WARMUP),
        '--repeat', str(REPEAT),
        '--trim', str(TRIM),
        '--update-pct', str(update_pct),
        '--output-csv', str(output_csv),
    ], ROOT, quiet=True)
    for line in result.stderr.splitlines():
        if '[iter' in line or 'Average' in line or 'history_' in line or 'fresh_' in line:
            print(' ', line)


def run_trace_point(table_type, repair_mode, historical_pct, delta_pct, output_csv):
    updates_file = ensure_update_file(FIXED_UPDATE_PCT)
    join_txs, scan_txs, delta_txs = trace_shape_from_delta_pct(delta_pct)
    historical_ratio = historical_pct / 100.0
    result = run_checked([
        str(TRACE_BIN),
        '--part-file', str(PART_FILE),
        '--lineitem-file', str(LINEITEM_FILE),
        '--updates-file', str(updates_file),
        '--table-type', table_type,
        '--repair-mode', repair_mode,
        '--bucket-num', str(BUCKET_NUM),
        '--worker-threads', str(TRACE_WORKER_THREADS),
        '--join-txs', str(join_txs),
        '--scan-txs', str(scan_txs),
        '--delta-txs', str(delta_txs),
        '--update-waves', str(TRACE_UPDATE_WAVES),
        '--readable-every', str(TRACE_READABLE_EVERY),
        '--historical-ratio', str(historical_ratio),
        '--warmup', str(WARMUP),
        '--repeat', str(REPEAT),
        '--trim', str(TRIM),
        '--update-pct', str(FIXED_UPDATE_PCT),
        '--output-csv', str(output_csv),
    ], ROOT, quiet=True)
    for line in result.stderr.splitlines():
        if '[iter' in line or 'Average' in line or 'total_ms:' in line:
            print(' ', line)

BLOCKING_CSV = DATA_DIR / 'sigmod_exp4_blocking.csv'
if BLOCKING_CSV.exists():
    BLOCKING_CSV.unlink()
for table_type, repair_mode in SERIES:
    print(f'blocking: {table_type}/{repair_mode}')
    run_blocking_point(table_type, repair_mode, FIXED_READER_THREADS, FIXED_UPDATE_PCT, BLOCKING_CSV)

df_block = normalize_result_df(pd.read_csv(BLOCKING_CSV))
display(df_block[[
    'table_type', 'repair_mode',
    'history_avg_read_wait_ms', 'history_avg_read_latency_ms',
    'fresh_avg_read_wait_ms', 'fresh_avg_read_latency_ms',
]])


In [ ]:
TRACE_FIXED_CSV = DATA_DIR / 'sigmod_exp4_trace_fixed.csv'
if TRACE_FIXED_CSV.exists():
    TRACE_FIXED_CSV.unlink()
for table_type, repair_mode in SERIES:
    print(f'trace fixed: {table_type}/{repair_mode}')
    run_trace_point(table_type, repair_mode, TRACE_FIXED_HIST_PCT, TRACE_FIXED_DELTA_PCT, TRACE_FIXED_CSV)

df_trace_fixed = normalize_result_df(pd.read_csv(TRACE_FIXED_CSV))
display(df_trace_fixed[['table_type', 'repair_mode', 'historical_pct', 'delta_pct', 'readable_every', 'total_ms', 'avg_join_latency_ms', 'avg_scan_latency_ms', 'avg_delta_latency_ms']])

TRACE_SWEEP_CSV = DATA_DIR / 'sigmod_exp4_trace_hist_sweep.csv'
if TRACE_SWEEP_CSV.exists():
    TRACE_SWEEP_CSV.unlink()
for hist_pct in HIST_PCT_SWEEP:
    for table_type, repair_mode in SERIES:
        print(f'trace hist sweep: hist_pct={hist_pct} {table_type}/{repair_mode}')
        run_trace_point(table_type, repair_mode, hist_pct, HIST_SWEEP_DELTA_PCT, TRACE_SWEEP_CSV)

df_trace_sweep = normalize_result_df(pd.read_csv(TRACE_SWEEP_CSV))
display(df_trace_sweep[['table_type', 'repair_mode', 'historical_pct', 'total_ms']].head())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.0))

labels = []
trace_values = []
trace_colors = []
for key, (label, color, linestyle, marker) in STYLE.items():
    table_type, repair_mode = key
    sub = df_trace_fixed[(df_trace_fixed['table_type'] == table_type) & (df_trace_fixed['repair_mode'] == repair_mode)]
    if sub.empty:
        raise ValueError(f'Missing trace fixed row for {table_type}/{repair_mode}')
    row = sub.iloc[0]
    labels.append(label)
    trace_values.append(float(row['total_ms']))
    trace_colors.append(color)

axes[0].bar(labels, trace_values, color=trace_colors, edgecolor='black', linewidth=0.4)
axes[0].set_title('Fixed 100-Tx HTAP Trace')
axes[0].set_ylabel('Total Completion Time (ms)')
axes[0].tick_params(axis='x', rotation=25)
axes[0].grid(True, axis='y', linestyle='--', linewidth=0.6, alpha=0.6)

for key, (label, color, linestyle, marker) in STYLE.items():
    table_type, repair_mode = key
    sub = df_trace_sweep[(df_trace_sweep['table_type'] == table_type) & (df_trace_sweep['repair_mode'] == repair_mode)].sort_values('historical_pct')
    if sub.empty:
        raise ValueError(f'Missing trace sweep rows for {table_type}/{repair_mode}')
    axes[1].plot(sub['historical_pct'], sub['total_ms'], color=color, linestyle=linestyle, marker=marker, linewidth=1.8, markersize=5, label=label)
axes[1].set_title('Trace vs Historical Mix')
axes[1].set_xlabel('Historical Read Percentage (%)')
axes[1].set_ylabel('Total Completion Time (ms)')
axes[1].grid(True, linestyle='--', linewidth=0.6, alpha=0.6)

handles, legend_labels = axes[1].get_legend_handles_labels()
fig.legend(handles, legend_labels, loc='upper center', ncol=5, bbox_to_anchor=(0.5, 1.10), framealpha=0.95)
fig.tight_layout(rect=[0, 0, 1, 0.92])
out_pdf = FIGS_DIR / 'sigmod_exp4_concurrency.pdf'
fig.savefig(out_pdf, format='pdf')
plt.show()
print('Saved', out_pdf)
